# Chart 4: the waiting list and the survey on one time axis
**Alessandro** · Data Visualisation 4 · The Foreigners Team

Question (Q3): did the hospital waiting list and the share of people who went without care because of waiting move the same way?

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PREPARED = Path("data/prepared")
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

ORANGE = "#eb6834"  # waiting
BLUE = "#2a78d6"    # cost
GREY = "#898781"    # EU average and background

unmet = pd.read_csv(PREPARED / "unmet_need_by_reason_v1_2026-11-11.csv")
waiting = pd.read_csv(PREPARED / "waiting_list_national_v1_2026-11-11.csv", parse_dates=["snapshot"])

: 

## Why this chart type
The data is two different measures over 2019 to 2025: the yearly average waiting list (people) and the share of people who said waiting stopped them getting care (%). I chose two stacked line panels on one time axis. A dual-axis chart would let me pick axis ranges that make the two lines look tightly linked, which overstates the connection. A connected scatter would hide time and is hard to read with only 7 points.

## Step 1: yearly average of the waiting list
Add the bands for each monthly snapshot, then average by year. 2026 is left out because it only has nine months.

In [ ]:
total = waiting.groupby("snapshot")["people"].sum() / 1000
yearly_list = total.groupby(total.index.year).mean().loc[2019:2025]
yearly_list.round(1)

## Step 2: the survey answer for Ireland

In [ ]:
ie = unmet[(unmet["group"] == "Ireland") & (unmet["reason"] == "Waiting list")]
survey = ie.set_index("year")["pct"].loc[2019:2025]
breaks = ie.set_index("year")["is_break"].loc[2019:2025]
survey

## Step 3: check the numbers against the guide

In [ ]:
assert round(yearly_list[2019] * 1000) == 555628
assert round(yearly_list[2021] * 1000) == 636015
assert round(yearly_list[2024] * 1000) == 585630
assert round(yearly_list[2025] * 1000) == 602124
assert [survey[y] for y in (2019, 2021, 2024, 2025)] == [1.0, 2.0, 2.7, 2.1]
print("All checks passed")

## Step 4: how often did they move the same way?

In [ ]:
same = (yearly_list.diff().dropna() > 0) == (survey.diff().dropna() > 0)
print(same.to_string())
print(f"Same direction in {same.sum()} of {len(same)} year changes")

## Step 5: the chart

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, sharex=True, figsize=(8, 5.5))
fig.suptitle("The waiting list and the survey answer moved the same way\noverall, but not every year",
             x=0.01, ha="left", fontsize=13, fontweight="bold", y=0.99)

# top: list
top.plot(yearly_list.index, yearly_list.values, color=ORANGE, marker="o", lw=2.2)
top.set_ylim(530, 670); top.set_ylabel("Waiting list\n(thousands of people)", fontsize=9)
for y, v in yearly_list.items():
    top.annotate(f"{v:.0f}", (y, v), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, color="#333")
top.annotate("List peaks in 2021", (2021, yearly_list[2021]), xytext=(2022.0, 658), fontsize=8.5,
             color=ORANGE, fontweight="bold", arrowprops=dict(arrowstyle="-", color=ORANGE, lw=0.8))
top.text(0.995, 0.04, "y-axis starts at 530", transform=top.transAxes, ha="right", fontsize=7.5, color=GREY)

# bottom: survey, hollow markers for break years
bottom.plot(survey.index, survey.values, color=ORANGE, lw=2.2, zorder=2)
for y, v in survey.items():
    hollow = bool(breaks[y])
    bottom.plot(y, v, "o", ms=7, mfc="white" if hollow else ORANGE, mec=ORANGE, mew=2, zorder=3)
    bottom.annotate(f"{v:.1f}%", (y, v), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, color="#333")
bottom.annotate("Survey answer peaks in 2024", (2024, survey[2024]), xytext=(2021.3, 2.95), fontsize=8.5,
                color=ORANGE, fontweight="bold", arrowprops=dict(arrowstyle="-", color=ORANGE, lw=0.8))
bottom.text(2019.05, 0.2, "Hollow points: break in the survey series", fontsize=7.5, color=GREY)
bottom.set_ylim(0, 3.3); bottom.set_ylabel("Went without care because\nof waiting (% of people 16+)", fontsize=9)
bottom.yaxis.set_major_formatter(lambda v, _: f"{v:.0f}%")
bottom.set_xticks(range(2019, 2026)); bottom.set_xlabel("Year (yearly average for the list; survey value for the answer)", fontsize=8.5)

for ax in (top, bottom):
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#e5e4e0", lw=0.6); ax.tick_params(labelsize=8.5)

fig.text(0.01, 0.012,
 "Source: NTPF outpatient waiting list open data; Eurostat EU-SILC (hlth_silc_08), accessed Nov 2026. Different measures: the list covers hospital outpatients,\n"
 "the survey covers all medical care. Shows direction, not cause. 2021 averages 11 snapshots (June missing). Stops at 2025: 2026 has only nine months\n"
 "(its average so far, about 666,000, is higher than 2021).", fontsize=6.6, color=GREY, va="bottom")
fig.tight_layout(rect=(0, 0.085, 1, 0.95))
fig.savefig(FIGURES / "chart4_list_and_survey.png", dpi=200)
plt.show()

## Commentary
Over 2019 to 2025 both measures end higher than they started (list 556k to 602k, survey answer 1.0% to 2.1%). Year by year the link is weaker: the list peaked in 2021 and fell until 2024, while the survey answer kept rising to 2.7% in 2024. They moved the same way in 3 of 6 year changes. The chart shows direction, not cause, and the two are different measures.

## Decisions
- Chart type: 
Two stacked line panels, one shared time axis. Rejected: dual axis (overstates the link) and connected scatter (hides time, only 7 points). My reason: The question is whether two different measures move the same way. Two panels let each one keep its own honest scale on a shared year axis, so the reader can compare direction without me choosing axis ranges that make the link look tighter. Bars would also have been wrong for the list, because from a zero baseline the change from 556k to 636k would look tiny.
- Title: 
"Moved the same way overall" is true because both end higher in 2025 than in 2019 (list 556k to 602k, survey 1.0% to 2.1%). "But not every year" is true because they matched in only 3 of 6 year changes, and from 2021 to 2024 the list fell while the survey answer rose from 2.0% to 2.7%. I kept it cautious because they are different measures and seven points can't show a cause.
- Annotated:
the 2021 list peak, the 2024 survey peak and are labelled because they fall in different years, and that mismatch is the main point of the chart. The hollow points mark 2019 and 2020 as survey break years, the same way as on the other charts, so the reader doesn't treat the jump from 2020 to 2021 as fully comparable.
- Left out:
For 2026 (nine months only) because 2026 only covers January to September, so its average isn't comparable to full years. I mention in the note that the 2026 average so far (about 666k) is higher than 2021, so leaving it out doesn't hide that.
- AI suggested / I decided: 
The AI set out four chart options with pros and cons (stacked panels, dual axis, connected scatter, indexed lines) and displayed pros and cons of each. From that I chose stacked panels and gave the reasons for rejecting the dual axis and the scatter. I then asked the AI to build the full chart and notebook as a draft for inspiration, and I plan to review it and edit it myself. The AI wrote the code and the annotation positions. I will decide the final title, annotations and wording.